In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import brain_probability_flux.plotting as bpp

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
max_fig_width = bpp.FIG_WIDTH

CLUSTER_SIZES = [4, 5, 6]
NUM_TASKS = 8
TASK_NAMES = bpp.TASK_NAMES_SHORT

OUT_DIR = Path("../figures/fig-s14")

In [ ]:
# Asymmetry ratio for K = 4, 5, 6 clusters (the tree cut as in fig-s07) with
# its trajectory bootstrap (scripts/bootstrap_asymmetry.py)
asymmetry = bpp.load_results("asymmetry_bootstrap", data_dir=DATA_DIR)
results = {
    int(K): {"mean": ratio, "std": boot.std(axis=1)}
    for K, ratio, boot in zip(
        asymmetry["num_clusters"], asymmetry["ratio"], asymmetry["ratio_boot"]
    )
}

In [ ]:
# 7-cluster results from the precomputed files used in Fig. 5f.
beta_J_k7 = bpp.load_results("inferred_networks", "arrhenius", DATA_DIR)[
    "beta_J"
]  # (8,7,7)
beta_J_boot_k7 = bpp.load_results(
    "inference_bootstrap", "arrhenius", DATA_DIR
)["beta_J"]  # (8, n_boot, 7, 7)
results[7] = {
    "mean": bpp.asymmetry_ratio(beta_J_k7),
    "std": bpp.asymmetry_ratio(beta_J_boot_k7).std(axis=1),
}
print(f"7 clusters: {np.round(results[7]['mean'], 3)}")

In [ ]:
ALL_K = [4, 5, 6, 7]
K_arr = np.array(ALL_K)
x_pos = np.arange(NUM_TASKS)
task_colors = bpp.task_colors_stepped(NUM_TASKS)
ebar_kw_bar = {
    "fmt": "none",
    "color": "k",
    "capsize": 2,
    "elinewidth": plt.rcParams["xtick.major.width"],
    "capthick": plt.rcParams["xtick.major.width"],
}
ebar_kw_line = {
    "capsize": 2,
    "elinewidth": plt.rcParams["xtick.major.width"],
    "capthick": plt.rcParams["xtick.major.width"],
}

OUT_DIR.mkdir(parents=True, exist_ok=True)

fig_a, (ax_a0, ax_a1, ax_a2) = plt.subplots(
    1,
    3,
    figsize=(3 * max_fig_width / 4, 65 / 25.4),
    sharey=True,
)

for idx, (K, ax) in enumerate(zip(CLUSTER_SIZES, [ax_a0, ax_a1, ax_a2])):
    means = results[K]["mean"]
    stds = results[K]["std"]
    ax.bar(x_pos, means, color=task_colors)
    ax.errorbar(x_pos, means, yerr=stds, **ebar_kw_bar)
    ax.axhline(
        means.mean(),
        color="k",
        ls="--",
        lw=plt.rcParams["xtick.major.width"],
        label=f"Mean = {means.mean():.3f}",
    )
    ax.set_xticks(x_pos)
    ax.set_xticklabels(TASK_NAMES, rotation=90)
    ax.set_title(f"{K} clusters")
    ax.legend(frameon=False, loc="upper left")
    if idx == 0:
        ax.set_ylabel(
            r"$\|\boldsymbol{J}^{(\mathrm{a})}\|_{\mathrm{F}}\,/\,"
            r"\|\boldsymbol{J}^{(\mathrm{s})}\|_{\mathrm{F}}$"
        )
    else:
        ax.tick_params(labelleft=False)

bpp.save_panel(fig_a, OUT_DIR, "a", bbox_inches="tight")

In [ ]:
fig_b, ax_b = plt.subplots(figsize=(max_fig_width / 4, 65 / 25.4))

for t, name in enumerate(TASK_NAMES):
    means_b = [results[K]["mean"][t] for K in ALL_K]
    stds_b = [results[K]["std"][t] for K in ALL_K]
    ax_b.errorbar(
        K_arr,
        means_b,
        yerr=stds_b,
        marker="o",
        ms=3,
        label=name,
        color=task_colors[t],
        **ebar_kw_line,
    )

ax_b.set_xlabel("Number of clusters")
ax_b.set_ylabel(
    r"$\|\boldsymbol{J}^{(\mathrm{a})}\|_{\mathrm{F}}\,/\,"
    r"\|\boldsymbol{J}^{(\mathrm{s})}\|_{\mathrm{F}}$"
)
ax_b.set_xticks(K_arr)
ax_b.legend(frameon=False, loc="lower right", fontsize=5, ncol=2)

bpp.save_panel(fig_b, OUT_DIR, "b", bbox_inches="tight")